In [4]:
# 필요한 라이브러리 설치 (tqdm이 없다면)
# !pip install tqdm

import os
import sys
import subprocess
from tqdm import tqdm

# --- 1. 기본 설정 ---
# ⚠️ 중요: 영상 파일이 있는 실제 경로로 변경하세요!
video_dir = r"C:\Temp\d1\Dreaming_of_Tomorrow"

# --- 2. 저장 형식 선택 (MP3 또는 WAV) ---
# 원하는 형식의 주석(#)을 해제하고, 원하지 않는 형식을 주석 처리하세요.

# (옵션 1: MP3로 저장)
#output_format = "mp3"
#output_audio_dir = os.path.join(video_dir, "audio_only_mp3")
#ffmpeg_command = [
#    '-vn',                # 비디오 제외
#    '-acodec', 'libmp3lame', # MP3 코덱
#    '-q:a', '2',          # VBR 품질 (0=최고, 9=최저, 2=고품질)
#]

# (옵션 2: WAV로 저장) - 용량이 큽니다.
output_format = "wav"
output_audio_dir = os.path.join(video_dir, "audio_only_wav")
ffmpeg_command = [
     '-vn',                # 비디오 제외
     '-acodec', 'pcm_s16le', # 표준 16비트 WAV 코덱 (원본 품질)
]

# --- 3. 디렉토리 설정 ---
if not os.path.exists(video_dir):
    print(f"❌ 비디오 디렉토리가 존재하지 않습니다: {video_dir}")
    print("스크립트의 'video_dir' 변수 값을 올바른 경로로 수정해주세요.")
    sys.exit(1)

# 출력 디렉토리 생성
os.makedirs(output_audio_dir, exist_ok=True)

# --- 4. 폴더 내 영상 파일 탐색 ---
try:
    video_files = [f for f in os.listdir(video_dir)
                   if f.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.ts'))]
    if not video_files:
        print(f"❌ {video_dir} 폴더에서 영상 파일을 찾을 수 없습니다.")
        sys.exit(1)
    print(f"📁 발견된 영상 파일: {len(video_files)}개 ({output_format} 오디오 추출 시작)")

except Exception as e:
    print(f"❌ 파일 탐색 중 오류 발생: {e}")
    sys.exit(1)

# --- 5. 오디오 추출 실행 ---
for video_file in tqdm(video_files, desc=f"🎧 Extracting {output_format}"):
    try:
        video_path = os.path.join(video_dir, video_file)
        base_name = os.path.splitext(video_file)[0]
        
        # 최종 오디오 파일 경로
        audio_path = os.path.join(output_audio_dir, f"{base_name}.{output_format}")

        # 이미 변환된 파일이 있는지 확인 (건너뛰기)
        if os.path.exists(audio_path):
            continue

        # FFmpeg 명령어 조합
        command = [
            'ffmpeg',
            '-i', video_path,
            *ffmpeg_command,  # 선택된 코덱 옵션 추가
            '-y',             # 덮어쓰기
            audio_path
        ]

        # FFmpeg 실행 (로그 숨김)
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    except subprocess.CalledProcessError:
        print(f"   ❌ 오류 ({video_file}): 오디오 트랙을 추출할 수 없습니다. (영상에 소리가 없는지 확인하세요)")
    except Exception as e:
        print(f"   ❌ 오류 ({video_file}): {str(e)}")

print("\n======================================")
print("🎉 모든 영상의 오디오 추출이 완료되었습니다!")
print(f"📂 오디오 파일 위치: {output_audio_dir}")

📁 발견된 영상 파일: 2개 (wav 오디오 추출 시작)


🎧 Extracting wav:   0%|          | 0/2 [00:00<?, ?it/s]

   ❌ 오류 (Dreaming_of_Tomorrow.f137.mp4): 오디오 트랙을 추출할 수 없습니다. (영상에 소리가 없는지 확인하세요)


🎧 Extracting wav: 100%|██████████| 2/2 [00:00<00:00,  7.14it/s]


🎉 모든 영상의 오디오 추출이 완료되었습니다!
📂 오디오 파일 위치: C:\Temp\d1\Dreaming_of_Tomorrow\audio_only_wav
